# Build a Recommender

**Week 4 · Lesson 8** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week4/lectures/08_build_a_recommender.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Setup

Same boilerplate as every notebook this term — see Week 1 Lesson 8 if this is your
first time seeing it.

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib scipy implicit git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

## 1. Business Problem

**Business question:** which products should the store surface to a customer — a
"customers who bought this also bought" panel on a product page, or a short
personalized list in a marketing email — to grow basket size and repeat visits,
using nothing but the pattern of who bought what?

**Why it matters:** Lessons 1–7 built every piece of this in isolation — a
distance metric and kNN (Lessons 1–3), the user-item matrix and two ways to
search it (Lesson 5), a fix for sparsity and cold-start (Lesson 6), and a way to
score a ranked list instead of a single prediction (Lesson 7). None of it has
touched real transactions yet. This notebook runs the whole toolkit end-to-end
on the **Online Retail** dataset — a UK-based online gift/houseware retailer,
~540K invoice lines from December 2010 to December 2011 — and asks which
approach a merchandising team should actually ship.

## 2. Data

*Load, describe, and sanity-check the data before touching a model.*

One row per **invoice line** — a single product on a single order — not one row
per customer. `load_online_retail()` in `gbus564_coursework.utils.course_data` already drops the
~25% of lines with no `CustomerID` (guest/unattributed orders — no basis for a
customer row at all), removes cancelled orders and non-positive
quantity/price rows (returns and adjustments, not purchases), and parses
`InvoiceDate`.

In [ ]:
from gbus564_coursework.utils.course_data import load_online_retail

df = load_online_retail()
df.head()

In [ ]:
print(f"{len(df):,} invoice lines")
print(f"{df['CustomerID'].nunique():,} customers")
print(f"{df['StockCode'].nunique():,} distinct products")
print(f"{df['InvoiceNo'].nunique():,} invoices")
print(f"date range: {df['InvoiceDate'].min().date()} to {df['InvoiceDate'].max().date()}")

In [ ]:
top_items = (
    df.groupby("Description")["Quantity"].sum().sort_values(ascending=False).head(10)
)
top_items.plot.barh(title="Top 10 products by units sold")
plt.gca().invert_yaxis()
plt.xlabel("units sold")
plt.tight_layout()

### Data Prep

Lesson 5's **user-item matrix** — customers × items — is the object every method
below operates on. The cell values here are **binary**: purchased or not. Raw
quantity ranges from single gifts to bulk wholesale re-orders, so it's a noisy
signal of preference; presence/absence of a purchase is the reliable one, and
it's what most production implicit-feedback recommenders use. Built as a
`scipy.sparse` matrix rather than a dense pivot table — a dense 4,338 × 3,665
matrix would mostly be the zeros Lesson 6 already warned about.

Evaluating a recommender needs **ground-truth relevant items** (Lesson 7) that
the model never sees while training — and, per Lesson 7, you can't get them by
holding out whole customers: a customer with no training row is a cold-start
case, which matrix factorization can't score at all (Lesson 6). Instead, hide
each customer's **most recent** purchases — Lesson 7's temporal holdout — with
one calendar cutoff for everyone: every order from **1 November 2011** on is
**test** (about 20% of all purchase lines), the six weeks before that are
**validation**, and everything earlier is **train** — Week 2's three buckets.
One shared date does two jobs a per-customer cutoff can't: a basket is never
split across train and test (every line on an invoice shares its date), and no
model sees *anyone's* purchases from after the cutoff — exactly the position
the store is in on the day it ships a model.

Two choices define what counts as a hit:

- **Relevant = new to that customer.** A product counts only if the customer
  bought it in the window and had *never bought it before*. This notebook
  measures **discovery** — what to show someone next — not re-ordering, which
  is a different job (Section 6).
- **No history, no score.** A customer whose first purchase falls in the
  window is a cold-start case (Lesson 6); no collaborative-filtering method
  has anything to work from, so they aren't scored.

In [ ]:
from scipy.sparse import coo_matrix, csr_matrix

# Give every customer and product an integer index: the row/column it owns in the matrix
customer_cat = df["CustomerID"].astype("category")
item_cat = df["StockCode"].astype("category")
customers = customer_cat.cat.categories  # row index -> CustomerID
items = item_cat.cat.categories          # column index -> StockCode
item_names = df.drop_duplicates("StockCode").set_index("StockCode")["Description"]  # for readable output

# One (customer, product) cell per invoice line
row = customer_cat.cat.codes.to_numpy()
col = item_cat.cat.codes.to_numpy()
data = np.ones(len(df))

# Sparse matrix: stores only the filled cells, since almost every cell is blank
full = coo_matrix((data, (row, col)), shape=(len(customers), len(items))).tocsr()
full.sum_duplicates()  # repeat purchases of the same product collapse into one cell
full.data = np.ones_like(full.data)  # purchased at all, regardless of how many times


def bought(X, u):
    """The products customer u has a purchase of in matrix X (column indices)."""
    return X.indices[X.indptr[u]:X.indptr[u + 1]]


def product_name(i):
    return item_names.get(items[i], "?")


print(f"user-item matrix: {full.shape[0]:,} customers x {full.shape[1]:,} items")
print(f"fill rate: {full.nnz / (full.shape[0] * full.shape[1]):.2%}  (Lesson 6's sparsity, concretely)")

In [ ]:
# Two calendar cutoffs, shared by every customer
VAL_START, TEST_START = pd.Timestamp("2011-09-20"), pd.Timestamp("2011-11-01")

# One row per invoice line: which customer bought which product, and when
buys = pd.DataFrame({"u": row, "i": col, "date": df["InvoiceDate"].to_numpy()})
is_train = buys["date"] < VAL_START
is_val = buys["date"].between(VAL_START, TEST_START, inclusive="left")
is_test = buys["date"] >= TEST_START


# Turn a set of (customer, product) lines into a matrix the same shape as `full`
def to_matrix(cells):
    cells = cells.drop_duplicates(["u", "i"])  # purchased at all, regardless of how many times
    return csr_matrix((np.ones(len(cells)), (cells["u"], cells["i"])), shape=full.shape)


train = to_matrix(buys[is_train])              # for choosing k
train_val = to_matrix(buys[is_train | is_val])  # for the final fit


# What counts as a hit: products new to a customer who already has history
def new_products(window, seen):
    """For each customer with earlier history, the products they bought in
    this window but never before it."""
    hits = {}
    for u, products in window.groupby("u")["i"]:
        before = set(bought(seen, u))
        if not before:
            continue                           # no history = cold-start, not scored
        new = set(products) - before           # never bought before = discovery
        if new:
            hits[u] = new
    return hits


# Each window is judged against everything before it
val_sets = new_products(buys[is_val], seen=train)
test_sets = new_products(buys[is_test], seen=train_val)
eval_users = np.array(sorted(test_sets))  # customers scored on test

# Sanity checks cited later: split sizes, cold-start customers, share of re-orders
test_cells = buys.loc[is_test, ["u", "i"]].drop_duplicates()
returning = test_cells["u"].map(lambda u: len(bought(train_val, u)) > 0)
print(f"purchase lines: train {is_train.mean():.1%}, validation {is_val.mean():.1%}, test {is_test.mean():.1%}")
print(f"{test_cells['u'].nunique():,} customers bought in the test window; "
      f"{test_cells.loc[~returning, 'u'].nunique():,} of them had no earlier history (cold-start)")
print(f"{1 - sum(map(len, test_sets.values())) / returning.sum():.0%} of returning customers' "
      f"test-window products were re-orders (not scored)")

## 3. Model

*Baseline first, then Lesson 5's two collaborative-filtering families.*

**Popularity** — recommend each customer whatever's best-selling overall, minus
what they've already bought — is this week's version of Week 2's mean/
majority-class baseline: no personalization at all, and every real method below
has to beat it to earn its complexity.

Popularity and item-based CF have nothing to tune, so they're fit straight on
**train + validation** — every purchase except the held-out test ones.

In [ ]:
item_popularity = np.asarray(train_val.sum(axis=0)).ravel()
pop_order = np.argsort(-item_popularity)


def recommend_popularity(u, k=10):
    already_bought = set(bought(train_val, u))
    return [i for i in pop_order if i not in already_bought][:k]

**Item-based CF (Lesson 5)** — cosine similarity between item *columns* (who
bought each item), then recommend items whose columns look like the columns of
what this customer already bought. Same kNN mechanism as Lesson 3, just pointed
at the transposed matrix.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

item_sim = cosine_similarity(train_val.T)


def recommend_item_cf(u, k=10):
    user_items = bought(train_val, u)
    if len(user_items) == 0:
        return recommend_popularity(u, k)
    scores = item_sim[user_items].sum(axis=0)
    scores[user_items] = -np.inf  # never recommend something already bought
    return np.argsort(-scores)[:k].tolist()

### Model Selection

**Matrix factorization (Lesson 6)** needs two choices memory-based CF doesn't.
The first is **how a blank cell counts** — Lesson 6's "real design choice":

- **Truncated SVD** (`sklearn`) treats every blank as a literal zero — a
  never-bought product reads as a vote *against* it.
- **Alternating least squares (ALS)** (`implicit`) keeps every blank but
  weights it far below an observed purchase — "unsure, weighted low," the
  treatment built for purchase data like this.

Both score a customer against a product with the dot product of their factor
vectors: Lesson 1's cosine without dividing by the lengths. The lengths stay in
because a product's vector length tracks how often it's bought.

The second is **how many latent factors, k**. The cell below fits both methods
on **train** only, at every k from 10 to 100 in steps of 10, and scores NDCG@10 against the
**validation** purchases — the test purchases stay untouched until Evaluate,
the same discipline Week 3 used to choose λ. (You *could* k-fold here —
shuffle the purchase rows into folds, not the customers — but over thousands
of customers one validation slice is already a stable read, and a
time-ordered split only has one "future" to fold on.)

Scoring uses Lesson 7's ranking metrics, defined once in the next cell and
reused unchanged in Evaluate.

In [ ]:
def precision_at_k(recommended, relevant, k):
    return len(set(recommended[:k]) & relevant) / k


def recall_at_k(recommended, relevant, k):
    return len(set(recommended[:k]) & relevant) / len(relevant) if relevant else 0.0


def ndcg_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    dcg = sum(1 / np.log2(i + 2) for i, item in enumerate(recommended_k) if item in relevant)
    ideal_hits = min(len(relevant), k)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0

In [ ]:
from sklearn.decomposition import TruncatedSVD
from implicit.als import AlternatingLeastSquares
from threadpoolctl import threadpool_limits

threadpool_limits(1, "blas")  # implicit runs its own threads; avoid oversubscribing

# Each fit_* returns a recommend(u) function, so both methods are scored the same way
def fit_svd(X, k):
    svd = TruncatedSVD(n_components=k, random_state=RANDOM_SEED)
    user_vecs = svd.fit_transform(X)   # one k-dim taste vector per customer
    item_vecs = svd.components_.T      # one k-dim vector per product

    def recommend(u, n=10):
        # dot product, not cosine: this customer's row of the rank-k reconstruction
        # of X, i.e. the model's predicted value for every product's cell
        scores = user_vecs[u] @ item_vecs.T
        scores[bought(X, u)] = -np.inf  # skip already bought
        return np.argsort(-scores)[:n].tolist()
    return recommend


def fit_als(X, k):
    # regularization and alpha (how much more an observed purchase counts than a
    # blank) were tuned on the same validation split; only k is swept here
    als = AlternatingLeastSquares(factors=k, regularization=10.0, alpha=1.0,
                                  iterations=20, random_state=RANDOM_SEED)
    als.fit(X, show_progress=False)

    def recommend(u, n=10):
        ids, _ = als.recommend(u, X[u], N=n, filter_already_liked_items=True)  # skip already bought
        return ids.tolist()
    return recommend


# Model selection: fit on train, score on validation, for each k
selection = {}
for name, fit in [("Truncated SVD", fit_svd), ("ALS", fit_als)]:
    for k in range(10, 101, 10):
        rec = fit(train, k)
        selection[(name, k)] = np.mean([ndcg_at_k(rec(u), val_sets[u], 10) for u in val_sets])

# One line per method; pick the k where validation NDCG@10 peaks
selection = pd.Series(selection, name="validation NDCG@10").unstack(0)
selection.plot(marker="o", title="Validation NDCG@10 by number of latent factors (k)")
plt.xlabel("k (latent factors)")
plt.ylabel("NDCG@10")
plt.tight_layout()

At small k the two methods are level: with 10 to 30 factors, neither has room
for much beyond the broad patterns, so how a blank counts barely matters. Past
that they split. Truncated SVD peaks at k=30 and then slides, because every
extra factor goes toward reconstructing the zeros, and to SVD every zero is a
"dislike." ALS keeps every blank but weights it low — Lesson 6's design choice —
and its **regularization** keeps extra factors from fitting noise, the same job
Week 3's λ did, so it holds steady and stays ahead from k=20 on.

ALS's curve is roughly flat past k=30, so its exact best k is partly noise.
Each final model uses its best validation k — **SVD at k=30, ALS at k=60** —
and both are now refit on **train + validation**. The validation purchases are
real data, and they've done their job. SVD stays in the comparison below so the
cost of the zero-as-dislike treatment shows up in the final scores too.

In [ ]:
recommend_svd = fit_svd(train_val, selection["Truncated SVD"].idxmax())
recommend_mf = fit_als(train_val, selection["ALS"].idxmax())

## 4. Evaluate

*Score against the baseline, not in isolation — Lesson 7's precision@k,
recall@k, and NDCG, computed the same way for all four methods, on the test purchases nothing
above has touched.*

In [ ]:
K = 10
methods = {
    "Popularity (baseline)": recommend_popularity,
    "Item-based CF": recommend_item_cf,
    "Matrix factorization — truncated SVD": recommend_svd,
    "Matrix factorization — ALS": recommend_mf,
}

rows = []
for name, fn in methods.items():
    for u in eval_users:
        relevant = test_sets[u]
        recommended = fn(u, K)
        rows.append({
            "method": name,
            "customer": u,
            "precision@10": precision_at_k(recommended, relevant, K),
            "recall@10": recall_at_k(recommended, relevant, K),
            "ndcg@10": ndcg_at_k(recommended, relevant, K),
        })
per_customer = pd.DataFrame(rows)

results = per_customer.groupby("method", sort=False)[["precision@10", "recall@10", "ndcg@10"]].mean()
results.round(3)

In [ ]:
results.plot.bar(rot=15, title=f"Recommendation quality on {len(eval_users):,} customers' later purchases")
plt.ylabel("score")
plt.tight_layout()

Every collaborative-filtering method clears the popularity baseline by a wide
margin, so personalization earns its keep. **ALS** posts the best average on all
three metrics, truncated SVD is second, and item-based CF trails both: the two
factorizations find more of the *new* products customers go on to buy than
direct co-purchase does. Averages, though, hide *who* each method serves.

Lesson 6 claimed matrix factorization's edge is customers with **sparse
histories**. The breakdown below checks that claim, scoring NDCG@10 separately
by how many products each customer had bought before their held-out
purchases.

In [ ]:
history = pd.Series({u: len(bought(train_val, u)) for u in eval_users})
per_customer["products bought before"] = pd.cut(
    per_customer["customer"].map(history), [0, 5, 10, 20, 50, 100, history.max()]
)

by_history = per_customer.pivot_table(
    index="products bought before", columns="method", values="ndcg@10", observed=True
)[list(methods)]
by_history["customers"] = per_customer.groupby("products bought before", observed=True)["customer"].nunique()
by_history.round(3)

ALS wins every segment, and its lead over truncated SVD is widest
for the **shortest** histories, where reading blanks as dislikes hurts most: a
customer with five purchases has thousands of blanks telling SVD what they
"don't want." That supports Lesson 6's claim — but check the `customers`
column before leaning on it. Few light buyers come back before a cutoff, so
the shortest-history rows rest on a few dozen customers each. That's a real
limit on testing the sparsity argument with this data, and worth knowing about
any dataset: the customers the claim is about are the hardest ones to evaluate.

Item-based CF trails both factorizations in every segment. "You bought one
piece, here's the rest of the set" is easy to explain, but learned factors find
patterns beyond direct co-purchase, and on genuinely new purchases that wins.

## 5. Interpret

*What is each method actually keying off of — and can a merchandiser trust the
"why"?*

In [ ]:
# Hand-picked for a clear theme (vintage Christmas); a typical customer gets 0-1 hits in 5
demo_u = customers.get_loc(16164)
past = bought(train_val, demo_u)
truth = test_sets[demo_u]


def show(i):
    return product_name(i) + (" ✓" if i in truth else "")  # ✓ = they went on to buy it


print("Already bought:", [product_name(i) for i in past])
print()
print("Item-based CF recommends:")
for i in recommend_item_cf(demo_u, 5):
    because = past[np.argmax(item_sim[past, i])]  # the past purchase most similar to this one
    print(f"  {show(i)}  <- because they bought {product_name(because)}")
print()
print("Matrix factorization (ALS) recommends:", [show(i) for i in recommend_mf(demo_u, 5)])
print()
print("Held-out truth (what they went on to buy):", [product_name(i) for i in truth])

Item-based CF's recommendations are directly explainable — every one of them is
"similar to something this customer already bought" in a literal, co-purchase
sense, the same as Amazon's "customers who bought this also bought." A
merchandiser can audit that reasoning. Matrix factorization's recommendations
come from arithmetic on **60 latent dimensions** nobody named — Lesson 6's
point exactly: those factors are learned to reconstruct the matrix well, not
designed to be legible. The cell below makes that same "customers who bought
this also bought" pattern concrete on the store's best-seller.

In [ ]:
demo_item = pop_order[0]
sims = item_sim[demo_item].copy()
sims[demo_item] = -np.inf
similar_items = [product_name(i) for i in np.argsort(-sims)[:5]]

print("Best-selling item:", product_name(demo_item))
print("Customers who bought it also bought:", similar_items)

Nothing here needed a product description or category — the similarity comes
entirely from co-occurrence in the matrix, Lesson 5's point that CF needs no
content features at all.

## 6. Recommend

**Recommendation:** ship **ALS matrix factorization** for personalized
discovery — the "you might also like" lists in marketing emails and on the
homepage. It finds the most of the new products customers went on to buy,
overall and in every history segment.

- Keep **item-based CF** for the on-site "customers who bought this also
  bought" panel. It's less accurate at predicting a customer's next new
  product, but it needs only the product being viewed, not a customer
  profile, and every recommendation is directly explainable to the
  merchandising team. Lesson 5's argument for item-based over user-based
  applies too: the product catalog is far more stable than the customer base,
  so item-item similarities stay valid longer between recomputes.
- Don't ship **truncated SVD**. It's the simplest factorization, but its
  zero-as-dislike treatment costs it accuracy against ALS on the same data.
- **Re-orders are a separate job.** This notebook scored discovery only, and
  roughly 40% of what returning customers bought in the test window were
  products they'd bought before. A plain "buy again" list of each customer's
  own frequent purchases covers that, alongside these recommendations rather
  than in competition with them.

No method solves true **cold-start**: the ~25% of transactions with no
`CustomerID`, and the brand-new customers who made their first purchase in the
test window (counted in the Data Prep output), get recommended from the same
**popularity** baseline this notebook used as its floor. That matches Lesson
6's honest caveat: matrix factorization narrows the hard problem to sparsity,
not zero-data cold-start.

Whichever ships, the deployed model is refit on **every** purchase, test
window included, and refit again on a schedule as new orders arrive. The scores
above are the honest estimate of how that model will do; nothing is re-scored
after the final refit.

- The full toolkit — user-item matrix, memory-based CF, model-based CF,
  ranking metrics — now runs end-to-end on real transactions, not a
  6-customer example
- **Evaluate the way you'll deploy**: one date cutoff, so no basket is split
  and no model sees the future — and know which job (discovery or re-ordering)
  you're scoring
- **How blanks count matters**: ALS beats truncated SVD on the same data
  because it treats a blank as "unsure," not "dislike"
- **ALS** wins on accuracy for every history segment; **item-based CF** keeps
  its place on explainability. Every method clears the popularity baseline
  by a wide margin
- **Cold-start remains unsolved** by every method — a popularity fallback is
  still required for customers with no history at all

**Next →** *k-means & Clustering Without Ground Truth — the same customers,
segmented without any label to predict.*